In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *

In [0]:
spark.conf.set(
    "fs.azure.account.key.stroadaccident01.dfs.core.windows.net",
    "<STORAGE_ACCOUNT_KEY>"
)

In [0]:
landing_path = "abfss://landing@stroadaccident01.dfs.core.windows.net/"
bronze_path = "abfss://bronze@stroadaccident01.dfs.core.windows.net/"

In [0]:
files = {
    "accident_details": "accident_details.csv",
    "location_details": "location_details.csv",
    "weather_details": "weather_details.csv",
    "road_features": "road_features.csv",
    "daylight_conditions": "daylight_conditions.csv"
}

In [0]:
import re

for table_name, file_name in files.items():

    print(f"Processing {table_name}")

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{landing_path}{file_name}")
    )

    # Clean column names
    for col_name in df.columns:
        clean_name = re.sub(r'[^A-Za-z0-9_]', '_', col_name)
        df = df.withColumnRenamed(col_name, clean_name)

    # Write Bronze Delta
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("mergeSchema", "true")
        .save(f"{bronze_path}{table_name}")
    )

    print(f"{table_name} completed")

In [0]:
for table_name in files.keys():

    df = spark.read.format("delta").load(f"{bronze_path}{table_name}")

    print("=" * 70)
    print(f"Table : {table_name}")
    print(f"Rows : {df.count()}")
    print(f"Columns : {len(df.columns)}")

    df.printSchema()

    display(df.limit(5))